# PerfectPunch v25 — MediaPipe Lite · Peak-Velocity Clips · DiNAT / ST-GCN · 5-Fold CV

**Goal:** Align the Python model *feature and clip contract* with the team's JavaScript website. The Python extractor uses MediaPipe Tasks **Pose Landmarker Lite (float16)** and 3D **world landmarks**, hip-centered / shoulder-width normalized. Skeletons are resampled to **60 FPS**, and each input is **50 frames ending at the smoothed compound-velocity peak** (including the peak). No reversal search is used.

**Segmentation contract:** 4 joints only — left/right elbows (13/14), left/right wrists (15/16); RMS of their 3D frame-to-frame displacement; **5-frame centered moving average** with zero padding; select the maximum after frame index 49 whenever at least 50 frames exist. If there are fewer than 50 detected/resampled frames, the Python offline extractor uses *first-frame front padding* to keep a training example; the JS app currently returns null for such short clips. See the deployment checklist at the end.

**Webcam robustness:** Training-only random approximation of **60→30→60 FPS** degradation, default probability 0.5. **Validation/test remain unaugmented.** An additional deterministic *webcam-proxy held-out evaluation* is saved for diagnosis; it is **not** equivalent to real JavaScript webcam captures (timing, pose tracking and clip detection can differ). This augmentation does not replace timestamp-aware JS preprocessing.

**Experiments:** Four candidates (`stgcn`, `stgcn_residual`, `tiny_dinat`, `dinat_residual`) run on the same 5 stratified video-level folds. Every fold has its own train/validation/test sets; folds are **not person-disjoint**. Evaluate before enabling full-data refit; old v21/v22/v23/v24 processed skeletons and checkpoints cannot be reused under the new peak-based preprocessing.

**Deployment:** Input float32 **`[1,7,50,4,3]`**, class order `jab`, `lead-hook`, `rear-uppercut`. The JavaScript team's `web/pipeline.js` still needs adjustment: `VEL_WINDOW=5`, `VELOCITY_LM=[13,14,15,16]`, and ch5 denominator `sqrt(vx²+vy²+vz²)`; this notebook **does not modify the website**. The current JS ONNX names `input`/`logits` differ from the notebook export names `skeleton_volume`/`class_logits` and must be reconciled. No Gradio code is included.


## 0. Install & mount Google Drive

In [1]:
!pip install -q "mediapipe>=0.10.30"   # 0.10.13 is no longer available on Colab
!pip install -q onnx onnxscript

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Global configuration — check these paths before running segmentation

In [2]:
import os, glob, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt

SEED = 42
CLIP_FRAMES = 50
TARGET_FPS = 60  # JS pipeline target FPS (30 FPS camera samples may be interpolated)
N_JOINTS = 33
CLASS_NAMES = ['jab', 'lead-hook', 'rear-uppercut']
N_CLASSES = len(CLASS_NAMES)
N_CHANNELS = 7
ARM_JOINTS = [13, 14, 15, 16]
N_ARM_JOINTS = len(ARM_JOINTS)
VELOCITY_LM = [13, 14, 15, 16]  # both elbows + both wrists, matching agreed JS contract
VEL_WINDOW = 5

# The team's shared cloud folder must first be accessible at this mounted path.
# Edit this line to match your Google Drive shortcut location.
POSE_VARIANT = 'lite'  # preprocessing AND live webcam; 'lite' | 'full' | 'heavy'
assert POSE_VARIANT in ('lite', 'full', 'heavy')
INPUT_FOLDER = '/content/drive/MyDrive/newPerfectPunchData/punch_dataset'
DRIVE_OUTPUT_BASE = '/content/drive/MyDrive/newPerfectPunchData/processed_data_lite_peak_v25'
DRIVE_NPY = os.path.join(DRIVE_OUTPUT_BASE, 'npy_data')
RUN_DIR = '/content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25'
CLASS_FOLDERS = {
    'jab': ['jab'],
    'lead-hook': ['hook', 'lead-hook'],
    'rear-uppercut': ['uppercut', 'rear-uppercut'],
}
VIDEO_EXTENSIONS = ('.mp4', '.mov', '.avi', '.m4v')

def class_folder(root, label):
    found = [Path(root)/name for name in CLASS_FOLDERS[label] if (Path(root)/name).is_dir()]
    if len(found) > 1:
        raise ValueError(f'Ambiguous source folders for {label}: {found}. Keep just one of the aliases.')
    return str(found[0]) if found else None

def source_videos(folder):
    return sorted([p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS],
                  key=lambda p: p.name.lower())

# Processing: False for quick re-runs after the .npy files exist.
AUTO_SEGMENT = True
FORCE_REPROCESS = False  # v25 uses a NEW folder; True to redo its existing .npy files

# Training-only augmentation. `v12` is identical to the uploaded source.
AUGMENTATION_MODE = 'v12'  # 'v12', 'improved', or 'none'
HOOK_SPECIFIC_AUG = False  # only in improved mode
MIRROR_PROB = 0.5          # only in improved mode
AUGMENT_TRAIN = True
# Training-only browser-style FPS degradation on segmented 50-frame clips.
# Kept separate from spatial/time-warp augmentation; NEVER applied to validation/test.
WEBCAM_FPS_AUG_PROB = 0.50
WEBCAM_CAPTURE_FPS = 30
assert 0 <= WEBCAM_FPS_AUG_PROB <= 1
PREPROCESS_CONTRACT = 'mediapipe_lite_peak4j_smooth5_60fps_50frames_ch5_3dspeed_v25'
SHORT_CLIP_POLICY = 'front_pad'  # JS currently returns null for <50 frames; see checklist
SAMPLER_MODE = 'hook_x3'   # 'hook_x3', 'balanced', 'shuffle'

N_SPLITS = 5  # True stratified 5-fold CV: ~20% test per fold, balanced by punch label
VAL_FRACTION = 0.15  # fraction of the remaining 80%, not of the full dataset
MAX_FOLDS = None  # None = all 5 folds; e.g. 2 for a quick debug run
MODEL_NAMES = ['tcn', 'tiny_cnn', 'stgcn', 'stgcn_residual', 'transformer', 'tiny_dinat', 'dinat_residual']
MODELS_TO_RUN = [
    'stgcn',           # unchanged 85k-param baseline
    'stgcn_residual',  # deeper residual graph model
    'tiny_dinat',      # unchanged 87k-param attention baseline
    'dinat_residual',  # enlarged, regularized DiNAT-style attention model
]

EPOCHS, PATIENCE = 120, 20
BATCH_SIZE, LR, WEIGHT_DECAY = 16, 3e-4, 1e-3
NUM_WORKERS = 0
RUN_EXPERIMENTS = True  # rerun all 5 held-out folds on the four candidates
FINAL_MODEL_NAME = 'dinat_residual'  # placeholder; choose AFTER reviewing complete CV metrics
RUN_FINAL_REFIT = True  # CV-only by default; enable after selecting final candidate

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
assert os.path.isdir(INPUT_FOLDER), f'Raw videos not found: {INPUT_FOLDER}. Check the shared Drive shortcut/path.'
os.makedirs(DRIVE_NPY, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)
for label in CLASS_NAMES:
    folder = class_folder(INPUT_FOLDER, label)
    print(f'{label:<15} <- {folder} ({len(source_videos(folder)) if folder else 0} videos)')
    if folder is None:
        raise FileNotFoundError(f'Missing source folder for {label}: expected one of {CLASS_FOLDERS[label]}')
print('Config ready | pose:', POSE_VARIANT, '| features:', N_CHANNELS, '| FPS:', TARGET_FPS, '| RAW:', INPUT_FOLDER)
print('Processed v25 peak clips:', DRIVE_NPY, '| v25 CV results:', RUN_DIR)

jab             <- /content/drive/MyDrive/newPerfectPunchData/punch_dataset/jab (40 videos)
lead-hook       <- /content/drive/MyDrive/newPerfectPunchData/punch_dataset/hook (40 videos)
rear-uppercut   <- /content/drive/MyDrive/newPerfectPunchData/punch_dataset/uppercut (40 videos)
Config ready | pose: lite | features: 7 | FPS: 60 | RAW: /content/drive/MyDrive/newPerfectPunchData/punch_dataset
Processed v25 peak clips: /content/drive/MyDrive/newPerfectPunchData/processed_data_lite_peak_v25/npy_data | v25 CV results: /content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25


## 2. MediaPipe Lite skeletons → 60 FPS → **peak-velocity** 50-frame clips

The raw video is passed through MediaPipe Tasks Pose Landmarker **Lite, VIDEO mode**, using world 3D coordinates. The original JS resampling convention (`linspace` endpoint interpolation) is preserved. After resampling, the compound speed of **four joints** is smoothed using a centered **5-frame** moving average. The clip consists of frames `[peak-49 : peak+1]` (end exclusive), including the peak. No search for reversal or retraction is performed. For source videos shorter than 50 resampled valid landmarks, the offline extractor pads at the front; check the mismatch with the JS `extractLiveClip` short-clip behavior.


In [3]:
import os
import cv2
import mediapipe as mp
import numpy as np
import glob

import urllib.request
from mediapipe.tasks.python import BaseOptions, vision

# mediapipe >= 0.10.30 removed mp.solutions, so pose runs through the Tasks
# PoseLandmarker. Keep extraction model consistent with the JS deployment.
POSE_TASK = f'/content/pose_landmarker_{POSE_VARIANT}.task'
if not os.path.exists(POSE_TASK):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
        f'pose_landmarker_{POSE_VARIANT}/float16/latest/'
        f'pose_landmarker_{POSE_VARIANT}.task', POSE_TASK)


def make_pose_landmarker():
    return vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=POSE_TASK),
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    ))

L_WRIST_IDX, R_WRIST_IDX = 15, 16
L_ELBOW_IDX, R_ELBOW_IDX = 13, 14


def normalize_skeleton(world_lm_list):
    """
    Hip-centred, shoulder-width-normalised from pose_world_landmarks.
    World frame: x=lateral, y=vertical (up), z=depth toward camera.
    Returns (33, 4) — [x, y, z, 1.0].
    """
    coords = np.array([[l.x, l.y, l.z] for l in world_lm_list], dtype=np.float32)
    hip_c  = (coords[23] + coords[24]) / 2.0
    coords -= hip_c
    dist   = np.linalg.norm(coords[11] - coords[12])
    if dist > 1e-6:
        coords /= dist
    vis = np.ones((33, 1), dtype=np.float32)
    return np.concatenate([coords, vis], axis=1)


def smooth_signal(arr, window=VEL_WINDOW):
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='same')


def resample_frames(frames, src_fps, dst_fps=TARGET_FPS):
    """Temporally resample skeleton frames from src_fps to dst_fps (copied from the app)."""
    n_src = len(frames)
    if n_src < 2 or abs(src_fps - dst_fps) < 0.5:
        return np.array(frames, dtype=np.float32)
    arr   = np.array(frames, dtype=np.float32)       # (N, 33, 4)
    dur   = n_src / src_fps
    n_dst = max(2, int(round(dur * dst_fps)))
    t_src = np.linspace(0, 1, n_src)
    t_dst = np.linspace(0, 1, n_dst)
    out   = np.zeros((n_dst, arr.shape[1], arr.shape[2]), dtype=np.float32)
    for j in range(arr.shape[1]):
        for c in range(arr.shape[2]):
            out[:, j, c] = np.interp(t_dst, t_src, arr[:, j, c])
    return out


def read_video_frames(video_path):
    """
    Read every detected-pose frame, then resample to TARGET_FPS.
    Returns (frames, TARGET_FPS).
    """
    cap     = cv2.VideoCapture(video_path)
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if src_fps <= 0 or src_fps > 500:
        src_fps = 30.0

    frames = []
    # Fresh landmarker per video: VIDEO mode needs increasing timestamps
    with make_pose_landmarker() as landmarker:
        i = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            rgb = np.ascontiguousarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            res = landmarker.detect_for_video(
                mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(i * 1000 / src_fps))
            i += 1
            if res.pose_world_landmarks and res.pose_landmarks:
                frames.append(normalize_skeleton(res.pose_world_landmarks[0]))
    cap.release()
    if len(frames) == 0:
        return [], TARGET_FPS
    return resample_frames(frames, src_fps, TARGET_FPS), TARGET_FPS


def find_peak_velocity_frame(frames):
    """Return the maximum smoothed four-joint speed, exactly 50-frame aligned.

    Use elbows 13/14 and wrists 15/16, RMS of per-frame 3D displacements,
    and centered zero-padded 5-frame moving average. Like the agreed JS
    algorithm, for >=50 frames only consider peaks at or after index 49.
    Ties choose the earliest index. No motion reversal is searched.
    """
    xyz = np.asarray(frames, dtype=np.float32)[:, :, :3]
    n = len(xyz)
    if n < 1:
        raise ValueError('Cannot find peak from empty sequence')
    movement = np.diff(xyz[:, VELOCITY_LM, :], axis=0, prepend=xyz[:1, VELOCITY_LM, :])
    speed = np.sqrt(np.mean(np.sum(movement**2, axis=-1), axis=1))
    # Explicit centered zero-padding for consistent n even for short sequences.
    pad = VEL_WINDOW // 2
    smooth = np.convolve(np.pad(speed, (pad, pad)),
                         np.ones(VEL_WINDOW, dtype=np.float64) / VEL_WINDOW,
                         mode='valid')
    min_peak = (CLIP_FRAMES - 1) if n >= CLIP_FRAMES else 0
    peak = min_peak + int(np.argmax(smooth[min_peak:]))
    return peak, float(smooth[peak])


def extract_best_clip(video_path):
    """Segment 50 normalized 60-FPS frames ending AT the speed peak."""
    frames, fps = read_video_frames(video_path)
    if len(frames) == 0:
        print('      No valid pose; skipping')
        return None
    peak, peak_speed = find_peak_velocity_frame(frames)
    end = peak + 1
    start = end - CLIP_FRAMES
    short = max(0, -start)
    if short:
        if SHORT_CLIP_POLICY != 'front_pad':
            raise ValueError(f'Unsupported short-clip policy: {SHORT_CLIP_POLICY}')
        padded = np.repeat(np.asarray(frames[:1], dtype=np.float32), short, axis=0)
        clip = np.concatenate([padded, np.asarray(frames[:end], dtype=np.float32)], axis=0)
    else:
        clip = np.asarray(frames[start:end], dtype=np.float32)
    assert clip.shape == (CLIP_FRAMES, N_JOINTS, 4), clip.shape
    print(f'      fps={fps:.1f}  peak@{peak}  speed={peak_speed:.5f}  '
          f'window[{max(0,start)}:{end}]  front_pad={short}  frames_total={len(frames)}')
    return clip


In [4]:
def segment_all(force=FORCE_REPROCESS):
    """One original video -> one .npy; unchanged extractor, explicit collision checks."""
    contract_file = Path(DRIVE_OUTPUT_BASE) / 'preprocessing_contract.json'
    expected = {
        'id': PREPROCESS_CONTRACT,
        'pose_variant': POSE_VARIANT,
        'fps': TARGET_FPS,
        'clip_frames': CLIP_FRAMES,
        'velocity_landmarks': VELOCITY_LM,
        'velocity_window': VEL_WINDOW,
        'selector': 'peak_inclusive',
        'short_clip_policy': SHORT_CLIP_POLICY,
    }
    cached = list(Path(DRIVE_NPY).glob('*/*.npy'))
    if contract_file.exists() and not force:
        current = json.loads(contract_file.read_text())
        if current != expected:
            raise RuntimeError(f'Cached clips have a different preprocessing contract. Set FORCE_REPROCESS=True. {contract_file}')
    elif cached and not force:
        raise RuntimeError(f'Existing clips without a v25 contract marker: {DRIVE_NPY}. Set FORCE_REPROCESS=True.')
    n_saved, n_reused, n_failed = 0, 0, 0
    for label in CLASS_NAMES:
        src = class_folder(INPUT_FOLDER, label)
        dst = Path(DRIVE_NPY) / label
        dst.mkdir(parents=True, exist_ok=True)
        videos = source_videos(src)
        stems = [p.stem.lower() for p in videos]
        if len(stems) != len(set(stems)):
            raise ValueError(f'Duplicate video stems in {src}; .mp4 and .mov can overwrite the same .npy')
        print(f'\n{label}: {len(videos)} source videos in {src}')
        for video in videos:
            target = dst / (video.stem + '.npy')
            if target.is_file() and not force:
                n_reused += 1
                continue
            clip = extract_best_clip(str(video))
            if clip is None:
                n_failed += 1
                print('  No detected skeleton:', video)
                continue
            assert clip.shape == (CLIP_FRAMES, N_JOINTS, 4), (video, clip.shape)
            np.save(target, clip.astype(np.float32))
            n_saved += 1
    if n_failed:
        raise RuntimeError(f'{n_failed} source videos had no valid MediaPipe pose. Fix/review those videos before five-fold CV.')
    contract_file.write_text(json.dumps(expected, indent=2))
    print(f'Segmentation: saved={n_saved} reused={n_reused} failed={n_failed} | {DRIVE_NPY}')

if AUTO_SEGMENT:
    segment_all()


jab: 40 source videos in /content/drive/MyDrive/newPerfectPunchData/punch_dataset/jab

lead-hook: 40 source videos in /content/drive/MyDrive/newPerfectPunchData/punch_dataset/hook

rear-uppercut: 40 source videos in /content/drive/MyDrive/newPerfectPunchData/punch_dataset/uppercut
Segmentation: saved=0 reused=120 failed=0 | /content/drive/MyDrive/newPerfectPunchData/processed_data_lite_peak_v25/npy_data


## 3. Feature engineering — exactly the uploaded seven channels

The `build_tensor()` below is **byte-for-byte the same Python function** as in `Copy_of_newPerfectPunchModel.ipynb`. It outputs `(7,T,4,3)` with position, velocity, geometry, y-velocity, elbow angle, lateral dominance, and vertical dominance. No model-specific feature changes; deployed inference uses this function too.

In [5]:
import numpy as np

L_SHOULDER, R_SHOULDER = 11, 12
L_ELBOW,    R_ELBOW    = 13, 14
L_WRIST,    R_WRIST    = 15, 16
L_HIP,      R_HIP      = 23, 24

# Tracked joint order: [L_elbow, R_elbow, L_wrist, R_wrist]  (indices 0-3)
TRACK    = [L_ELBOW, R_ELBOW, L_WRIST, R_WRIST]
SHOULDER = [L_SHOULDER, R_SHOULDER, L_SHOULDER, R_SHOULDER]


def angle_cos(a, b, c):
    ba = a - b; bc = c - b
    return float(np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8))


def build_tensor(clip):
    """
    clip: (T, 33, 4)  — world-3D skeleton, columns [x, y, z, vis]
                        x = lateral (left/right), y = vertical (up/down),
                        z = depth (toward camera), shoulder-width normalised.

    Returns float32 tensor of shape (7, T, 4, 3).

    Channel layout
    --------------
    ch0  position xyz          [L_elbow, R_elbow, L_wrist, R_wrist]
    ch1  velocity xyz          frame-to-frame displacement (world 3-D)
    ch2  geometry              bone unit vectors; x-component of elbow rows
                               replaced by elbow flexion angle cosine
    ch3  y-velocity            raw vertical motion per joint (scalar broadcast)
                               (hook: elbow rises; uppercut: wrist rockets up)
    ch4  elbow flexion angle   cos(angle) broadcast to all 4 joint rows
                               (jab ~+1.0 = extended; hook/uppercut ~0.0-0.5)
    ch5  lateral dominance     signed x_vel / ||vel||  — how sideways is the motion?
                               jab  ≈ 0  (forward/z dominant)
                               hook ≈ ±1 (x dominant)
                               uppercut ≈ 0 (y dominant)
    ch6  vertical dominance    signed y_vel / ||vel||  — how upward is the motion?
                               jab      ≈ 0  (z dominant)
                               hook     ≈ 0  (x dominant, stays level)
                               uppercut ≈ +1 (y dominant — wrist drives steeply up)
    """
    T   = clip.shape[0]
    xyz = clip[:, :, :3].astype(np.float32)   # (T, 33, 3)

    tracked = xyz[:, TRACK, :]   # (T, 4, 3)

    # ── ch0: position ────────────────────────────────────────────────────────
    ch0 = tracked.copy()

    # ── ch1: velocity (world 3-D frame-to-frame displacement) ────────────────
    ch1     = np.zeros_like(tracked)
    ch1[1:] = tracked[1:] - tracked[:-1]

    # ── ch2: geometry — bone unit vectors + elbow flexion angle ──────────────
    ch2 = np.zeros_like(tracked)
    for t in range(T):
        f = xyz[t]
        for i in range(4):
            if i < 2:   # elbows: shoulder → elbow
                v = f[TRACK[i]] - f[SHOULDER[i]]
            else:       # wrists: elbow → wrist
                v = f[TRACK[i]] - f[TRACK[i - 2]]
            ch2[t, i] = v / (np.linalg.norm(v) + 1e-8)
        # Overwrite x-component of elbow rows with flexion angle cosine
        ch2[t, 0, 0] = angle_cos(f[L_SHOULDER], f[L_ELBOW], f[L_WRIST])
        ch2[t, 1, 0] = angle_cos(f[R_SHOULDER], f[R_ELBOW], f[R_WRIST])

    # ── ch3: y-velocity — raw vertical motion scalar ─────────────────────────
    # Distinct from ch6 (normalised): raw magnitude matters here — a fast
    # uppercut has large absolute y_vel even before normalising.
    ch3   = np.zeros_like(tracked)
    y_vel = np.zeros((T, 4), dtype=np.float32)
    for i, lm in enumerate(TRACK):
        y_vel[1:, i] = xyz[1:, lm, 1] - xyz[:-1, lm, 1]
    ch3[:, :, :] = y_vel[:, :, np.newaxis]   # broadcast to 3 cols

    # ── ch4: elbow flexion angle, broadcast to all 4 joint rows ──────────────
    ch4 = np.zeros_like(tracked)
    for t in range(T):
        f = xyz[t]
        l_angle = angle_cos(f[L_SHOULDER], f[L_ELBOW], f[L_WRIST])
        r_angle = angle_cos(f[R_SHOULDER], f[R_ELBOW], f[R_WRIST])
        ch4[t, 0, :] = l_angle
        ch4[t, 1, :] = r_angle
        ch4[t, 2, :] = l_angle
        ch4[t, 3, :] = r_angle

    # ── ch5 & ch6: lateral and vertical dominance ────────────────────────────
    # Both use the same denominator (total speed) so they form an
    # interpretable decomposition of motion direction.
    #
    #              jab    hook   uppercut
    #   ch5 (x):   ≈ 0    ≈ ±1    ≈ 0
    #   ch6 (y):   ≈ 0    ≈  0    ≈ +1   ← unique uppercut signature
    #
    # Together they triangulate all 3 classes cleanly.
    eps   = 1e-8
    vel   = ch1                                                   # (T, 4, 3)
    speed = np.linalg.norm(vel, axis=-1, keepdims=True) + eps    # (T, 4, 1)

    lat_dom  = vel[:, :, 0:1] / speed   # x_vel / ||vel||  (T, 4, 1)
    vert_dom = vel[:, :, 1:2] / speed   # y_vel / ||vel||  (T, 4, 1)

    ch5 = np.broadcast_to(lat_dom,  (T, 4, 3)).copy()
    ch6 = np.broadcast_to(vert_dom, (T, 4, 3)).copy()

    return np.stack([ch0, ch1, ch2, ch3, ch4, ch5, ch6], axis=0).astype(np.float32)

assert build_tensor(np.zeros((CLIP_FRAMES, 33, 4), np.float32)).shape == (7, CLIP_FRAMES, 4, 3)
print('Feature shape:', (7, CLIP_FRAMES, 4, 3))

Feature shape: (7, 50, 4, 3)


## 4. Video manifest (not subject IDs)

Class folder names are remapped to canonical output folders: `hook/` → `lead-hook/`, `uppercut/` → `rear-uppercut/`. Every **original video** corresponds to exactly one `.npy` item in the manifest. It checks processed clip availability and ignores unrelated stale `.npy` files rather than silently introducing extra training data. No `subject_mapping.csv` is needed.

In [6]:
def build_manifest(raw_root=INPUT_FOLDER, npy_root=DRIVE_NPY):
    rows = []
    for class_id, label in enumerate(CLASS_NAMES):
        raw_dir = class_folder(raw_root, label)
        if raw_dir is None:
            raise FileNotFoundError(f'No raw folder for {label} (aliases: {CLASS_FOLDERS[label]})')
        videos = source_videos(raw_dir)
        if not videos:
            raise ValueError(f'No raw videos for {label}: {raw_dir}')
        stems = [p.stem.lower() for p in videos]
        if len(set(stems)) != len(stems):
            raise ValueError(f'Nonunique video stems for class {label}: {raw_dir}')
        for video in videos:
            processed = Path(npy_root) / label / (video.stem + '.npy')
            if not processed.is_file():
                raise FileNotFoundError(f'Missing processed clip for {video}: {processed}. Run segment_all().')
            clip = np.load(processed, mmap_mode='r')
            if clip.ndim != 3 or clip.shape[1:] != (33, 4) or clip.shape[0] < 1:
                raise ValueError(f'Bad clip {processed}: {clip.shape}, expected (T,33,4)')
            rows.append({'path': str(processed), 'video_path':str(video),
                         'video_id':f'{label}/{video.stem}',
                         'relative_path':f'{label}/{video.stem}.npy',
                         'label':int(class_id), 'class_name':label})
    df = pd.DataFrame(rows).reset_index(drop=True)
    assert df.video_id.is_unique
    df.to_csv(Path(RUN_DIR)/'video_manifest.csv', index=False)
    print('Video counts:',df.class_name.value_counts().reindex(CLASS_NAMES,fill_value=0).to_dict(), '| Total:',len(df))
    return df

# After segmentation: manifest = build_manifest()

## 5. Training augmentation and datasets

Retain the notebook's existing `v12` (or optional `improved`) spatial/temporal augmentation. Additionally, with probability `WEBCAM_FPS_AUG_PROB=0.5`, degrade the 50-frame clip from 60 FPS to approximately 30 FPS and linearly resample back to 60 FPS before building the seven-channel tensor. This **approximates** the live browser's loss of temporal resolution; it does not reproduce live MediaPipe tracking, irregular timestamps, or how peak detection might shift at 30 FPS. It is applied **only to training data**, never to standard validation/test data. A deterministic webcam-proxy held-out test is also reported separately to illuminate robustness. It should not replace a real webcam evaluation.

The JS app's four-joint speed peak / smoothing / ch5 requirements are recorded in the final deployment checklist. The offline `front_pad` policy for short videos is retained to avoid silently deleting labeled samples; the website currently returns `null` until it has at least 50 resampled frames.


In [7]:
def augment_clip(clip, label_name=None):
    """(T, 33, 4) -> augmented clip. Hooks get stronger augmentation."""
    clip     = clip.copy()
    xyz      = clip[:, :, :3]
    is_hook  = (label_name == 'lead-hook')

    # Mirror left <-> right
    if np.random.random() > 0.5:
        xyz[:, :, 0] = -xyz[:, :, 0]

    # Temporal speed warp
    T     = xyz.shape[0]
    warp  = 0.35 if is_hook else 0.25
    speed = np.random.uniform(1 - warp, 1 + warp)
    src   = np.linspace(0, T - 1, max(int(T * speed), T // 2))
    warped = np.array([xyz[min(int(i), T - 1)] for i in src])
    idx    = np.linspace(0, len(warped) - 1, T)
    xyz    = np.array([warped[min(int(i), len(warped) - 1)] for i in idx])

    # Yaw rotation (larger for hooks to cover more lateral angles)
    max_yaw = 40 if is_hook else 25
    ang = np.random.uniform(-max_yaw, max_yaw) * np.pi / 180
    c, s = np.cos(ang), np.sin(ang)
    R    = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]], dtype=np.float32)
    xyz  = xyz @ R.T

    # Scale jitter +-12%
    xyz *= np.random.uniform(0.88, 1.12)

    # Gaussian noise
    noise_std = 0.007 if is_hook else 0.005
    xyz += np.random.normal(0, noise_std, xyz.shape).astype(np.float32)

    # Hook-specific: pitch tilt to simulate elbow height variation
    if is_hook and np.random.random() > 0.4:
        pitch = np.random.uniform(-15, 15) * np.pi / 180
        cp, sp = np.cos(pitch), np.sin(pitch)
        Rp = np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]], dtype=np.float32)
        xyz = xyz @ Rp.T

    clip[:, :, :3] = xyz
    return clip


# Mirror pairs from MediaPipe's 33 landmark schema (excluding midline landmarks).
_MIRROR_PAIRS = [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
                 (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)]
_MIRROR_IDX = np.arange(33)
for a,b in _MIRROR_PAIRS:
    _MIRROR_IDX[a],_MIRROR_IDX[b] = b,a

def augment_clip_improved(clip, label_name=None):
    clip = clip.copy()
    xyz = clip[:,:,:3].copy()
    is_hook = HOOK_SPECIFIC_AUG and label_name == 'lead-hook'
    if np.random.rand() < MIRROR_PROB:
        xyz = xyz[:,_MIRROR_IDX,:].copy()
        xyz[:,:,0] *= -1

    # Resample the sequence at a randomized playback speed, retaining output T.
    T = len(xyz)
    speed = np.random.uniform(0.65 if is_hook else 0.75,
                              1.35 if is_hook else 1.25)
    center = (T - 1) / 2
    src = np.clip((np.arange(T)-center) * speed + center, 0, T-1)
    lo, hi = np.floor(src).astype(int), np.ceil(src).astype(int)
    a = (src - lo).astype(np.float32)[:,None,None]
    xyz = xyz[lo]*(1-a) + xyz[hi]*a

    yaw_deg = 40 if is_hook else 25
    ang = np.deg2rad(np.random.uniform(-yaw_deg, yaw_deg))
    c,s = np.cos(ang),np.sin(ang)
    yaw = np.array([[c,0,s],[0,1,0],[-s,0,c]],dtype=np.float32)
    xyz = xyz @ yaw.T
    xyz *= np.random.uniform(0.88,1.12)
    xyz += np.random.normal(0,0.007 if is_hook else 0.005,xyz.shape).astype(np.float32)
    if is_hook and np.random.rand() > 0.4:
        a = np.deg2rad(np.random.uniform(-15,15)); c,s = np.cos(a),np.sin(a)
        pitch = np.array([[1,0,0],[0,c,-s],[0,s,c]],dtype=np.float32)
        xyz = xyz @ pitch.T
    clip[:,:,:3] = xyz
    return clip



def apply_train_augmentation(clip, label_name):
    if AUGMENTATION_MODE == 'none': return clip
    if AUGMENTATION_MODE == 'v12': return augment_clip(clip, label_name)
    if AUGMENTATION_MODE == 'improved': return augment_clip_improved(clip, label_name)
    raise ValueError(f'Unknown AUGMENTATION_MODE: {AUGMENTATION_MODE}')

def prepare_raw_clip(raw):
    raw = np.asarray(raw, dtype=np.float32)
    if raw.ndim != 3 or raw.shape[1:] != (33,4) or len(raw) == 0:
        raise ValueError(f'Expected (T,33,4) skeleton, got {raw.shape}')
    if len(raw) < CLIP_FRAMES:
        padding = np.repeat(raw[:1], CLIP_FRAMES-len(raw), axis=0)
        raw = np.concatenate((padding,raw),axis=0)
    elif len(raw) > CLIP_FRAMES:
        # Extraction guarantees 50; fail on a stale/incorrect upstream clip.
        raise ValueError(f'Expected 50-frame segmented clip, got {len(raw)}. Reprocess with segment_all(force=True).')
    return raw

def simulate_webcam_30_to_60(clip, capture_fps=WEBCAM_CAPTURE_FPS):
    """Approximate sparse webcam sampling and JS-style linear reinterpolation.

    Preserve fixed (50,33,4) shape. The original 50-frame clip spans only
    about 0.83s; sampling its interior is a training-time proxy and does NOT
    exactly simulate re-running the peak detector after 30-FPS acquisition.
    """
    clip = np.asarray(clip, dtype=np.float32)
    if not (0 < capture_fps < TARGET_FPS):
        return clip.copy()
    n = len(clip)
    # Simulate a camera capturing positions at 0, 2, 4, ... for 30->60 FPS.
    src = np.arange(0, n, TARGET_FPS / float(capture_fps), dtype=np.float64)
    if len(src) < 2:
        return clip.copy()
    a = np.floor(src).astype(int)
    b = np.minimum(a + 1, n - 1)
    f = (src - a).astype(np.float32)[:, None, None]
    sparse = clip[a] * (1 - f) + clip[b] * f
    # Match browser's linspace-endpoint temporal resampler, keep EXACTLY 50.
    positions = np.linspace(0, len(sparse) - 1, n, dtype=np.float64)
    lo = np.floor(positions).astype(int)
    hi = np.minimum(lo + 1, len(sparse) - 1)
    frac = (positions - lo).astype(np.float32)[:, None, None]
    return (sparse[lo] * (1 - frac) + sparse[hi] * frac).astype(np.float32)


class BoxingDataset(Dataset):
    def __init__(self, manifest, augment=False, webcam_simulated=False):
        self.frame = manifest.reset_index(drop=True)
        self.augment = augment
        self.webcam_simulated = webcam_simulated
        if augment and webcam_simulated:
            raise ValueError('Do not mix random train augment with deterministic held-out webcam proxy')

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[int(idx)]
        raw = prepare_raw_clip(np.load(row.path))
        if self.augment:
            raw = apply_train_augmentation(raw, row.class_name)
            if WEBCAM_FPS_AUG_PROB and np.random.random() < WEBCAM_FPS_AUG_PROB:
                raw = simulate_webcam_30_to_60(raw)
        elif self.webcam_simulated:
            raw = simulate_webcam_30_to_60(raw)
        feat = build_tensor(raw)
        if not np.isfinite(feat).all():
            raise ValueError(f'Non-finite features in {row.path}')
        return torch.from_numpy(feat), torch.tensor(int(row.label), dtype=torch.long)


## 6. Stratified 5-Fold Cross-Validation (video-level)

Use `sklearn.model_selection.StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)` to allocate **whole videos** to five mutually exclusive test folds. The stratification variable is the **punch label**, not the person's identity or filename. With 40 videos per class, each fold's test set has exactly **8 jab + 8 hook + 8 uppercut = 24 videos**.

For each fold, take the remaining 96 videos and create an **independent stratified validation split** (`VAL_FRACTION=0.15`): **81 train / 15 val / 24 test** for the current balanced 120-video dataset. Test never enters training, early stopping, or checkpoint selection. No augmentation in validation or test. All architectures share the same fixed folds.

`MAX_FOLDS=2` runs only two of the five folds for a smoke test, and is **not** a complete cross-validation result. `MAX_FOLDS=None` evaluates all five folds, covering each video as test exactly once. Videos from the same **person** can appear in both training and test: this evaluates new recordings, not unseen people.


In [8]:
def iter_stratified5_folds(manifest, seed=SEED, max_folds=MAX_FOLDS):
    """Yield 5 stratified VIDEO-level test folds, with separate stratified validation.

    One video = one manifest row. No grouping by individual is performed.
    All videos are held out exactly once as test when max_folds is None.
    """
    manifest = manifest.reset_index(drop=True)
    n = len(manifest)
    labels = manifest.label.to_numpy(dtype=int)
    video_ids = manifest.video_id.astype(str).to_numpy()
    if len(set(video_ids)) != n:
        raise ValueError('video_id must be unique: do not treat clips from one video as independent samples.')
    counts = np.bincount(labels, minlength=N_CLASSES)
    if len(labels) == 0 or any(counts[c] < N_SPLITS for c in range(N_CLASSES)):
        raise ValueError(f'Stratified {N_SPLITS}-fold CV requires at least {N_SPLITS} videos per class; counts={counts.tolist()}')
    if not 0 < VAL_FRACTION < 1:
        raise ValueError(f'VAL_FRACTION must be between 0 and 1, got {VAL_FRACTION}')

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    split_pairs = list(skf.split(np.zeros(n), labels))
    if max_folds is not None and not (1 <= int(max_folds) <= N_SPLITS):
        raise ValueError(f'MAX_FOLDS must be 1..{N_SPLITS} or None; got {max_folds}')
    chosen = range(N_SPLITS if max_folds is None else int(max_folds))
    for fold_id in chosen:
        train_val_idx, test_idx = split_pairs[fold_id]
        try:
            train_idx, val_idx = train_test_split(
                train_val_idx,
                test_size=VAL_FRACTION,
                stratify=labels[train_val_idx],
                random_state=seed + 1000 + fold_id,
                shuffle=True,
            )
        except ValueError as exc:
            raise ValueError(
                f'Fold {fold_id}: not enough class examples for a separate stratified validation set. '
                f'Increase dataset size or VAL_FRACTION; original error: {exc}'
            ) from exc
        train_idx = np.sort(np.asarray(train_idx, dtype=int))
        val_idx = np.sort(np.asarray(val_idx, dtype=int))
        test_idx = np.sort(np.asarray(test_idx, dtype=int))
        train_set, val_set, test_set = set(train_idx), set(val_idx), set(test_idx)
        if train_set & val_set or train_set & test_set or val_set & test_set:
            raise AssertionError(f'Fold {fold_id}: train/val/test overlap.')
        if (train_set | val_set | test_set) != set(range(n)):
            raise AssertionError(f'Fold {fold_id}: missing videos.')
        if any(set(labels[idx]) != set(range(N_CLASSES)) for idx in (train_idx, val_idx, test_idx)):
            raise ValueError(f'Fold {fold_id}: one or more classes are missing in train/val/test.')
        yield {
            'fold': fold_id, 'train': train_idx, 'val': val_idx, 'test': test_idx,
            'test_video_ids': video_ids[test_idx].tolist(), 'n_test': len(test_idx),
        }


def check_folds(manifest, max_folds=MAX_FOLDS):
    """Print counts and audit uniqueness, stratification, and video overlap."""
    folds = list(iter_stratified5_folds(manifest, max_folds=max_folds))
    test_indices = [int(i) for f in folds for i in f['test']]
    assert len(test_indices) == len(set(test_indices)), 'A video appears in test more than once.'
    for f in folds:
        test_classes = manifest.iloc[f['test']].class_name.value_counts().reindex(CLASS_NAMES, fill_value=0).to_dict()
        print(f"Fold {f['fold']:03d} | train={len(f['train']):3d} "
              f"val={len(f['val']):3d} test={len(f['test']):3d} | "
              f"test_classes={test_classes}")
    print(f'Test coverage: {len(test_indices)}/{len(manifest)} unique videos across {len(folds)} folds.')
    if max_folds is None:
        assert len(folds) == N_SPLITS and sorted(test_indices) == list(range(len(manifest)))
    return folds

# After build_manifest(): check_folds(manifest)


## 7. Model zoo — all accept `(B,7,50,4,3)`

Compare **four models on identical folds**:

- `stgcn`: unchanged three-block graph-convolution baseline.
- `stgcn_residual`: six-block normalized graph/temporal residual architecture from v22.
- `tiny_dinat`: unchanged three-block, 48-channel DiNAT-style attention baseline.
- `dinat_residual`: **new** four-block, 96-channel DiNAT-style alternative with pre-LayerNorm on each branch, residual LayerScale, dropout, dilations `[1,2,4,8]`, and joint/temporal average + max pooling to retain peak punch dynamics.

The new attention model is intentionally moderate in size: 120 videos are too few to assume a much bigger model will generalize better. All versions preserve the 7-channel feature contract and class ordering for ONNX / JavaScript inference.


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Conv2Plus1DBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kt=3, kj=3, kc=3, stride_t=1, dropout=0.1):
        super().__init__()
        mid = max(in_ch, out_ch) // 2 if in_ch != out_ch else in_ch
        self.spatial = nn.Sequential(
            nn.Conv3d(in_ch, mid, kernel_size=(1, kj, kc),
                      padding=(0, kj//2, kc//2), bias=False),
            nn.BatchNorm3d(mid), nn.ReLU(inplace=True),
        )
        self.temporal = nn.Sequential(
            nn.Conv3d(mid, out_ch, kernel_size=(kt, 1, 1),
                      stride=(stride_t, 1, 1), padding=(kt//2, 0, 0), bias=False),
            nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True),
        )
        self.drop = nn.Dropout3d(dropout)
        self.residual = nn.Identity() if (in_ch == out_ch and stride_t == 1) else nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 1, stride=(stride_t, 1, 1), bias=False),
            nn.BatchNorm3d(out_ch)
        )

    def forward(self, x):
        return F.relu(self.drop(self.temporal(self.spatial(x))) + self.residual(x), inplace=True)


class PerfectPunchCNN(nn.Module):
    """
    6-block (2+1)D CNN for skeleton punch classification.
    Input:  (B, 7, T, 4, 3)   -- 7 channels, T frames, 4 arm joints, 3 coords
    Output: (B, num_classes)
    """
    def __init__(self, num_classes=3, in_channels=7, base_ch=32, dropout=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, base_ch, kernel_size=(3, 3, 3),
                      padding=(1, 1, 1), bias=False),
            nn.BatchNorm3d(base_ch), nn.ReLU(inplace=True),
        )
        self.layer1 = Conv2Plus1DBlock(base_ch,      base_ch * 2,  kt=3, dropout=0.10)
        self.layer2 = Conv2Plus1DBlock(base_ch * 2,  base_ch * 4,  kt=3, stride_t=2, dropout=0.10)
        self.layer3 = Conv2Plus1DBlock(base_ch * 4,  base_ch * 8,  kt=3, stride_t=2, dropout=0.15)
        self.layer4 = Conv2Plus1DBlock(base_ch * 8,  base_ch * 8,  kt=3, dropout=0.15)
        self.layer5 = Conv2Plus1DBlock(base_ch * 8,  base_ch * 16, kt=3, stride_t=2, dropout=0.20)
        self.layer6 = Conv2Plus1DBlock(base_ch * 16, base_ch * 16, kt=3, dropout=0.20)
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(base_ch * 16, base_ch * 8),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(base_ch * 8, num_classes),
        )

    def forward(self, x):   # x: (B, 6, T, 4, 3)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.layer6(x)
        x = self.pool(x)
        return self.classifier(x)

# 7x3 per joint = 21 features, x 4 joints = 84 features per timestep.
class TemporalBlock(nn.Module):
    def __init__(self, cin, cout, dilation=1, dropout=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(cin,cout,3,padding=dilation,dilation=dilation,bias=False),
            nn.BatchNorm1d(cout), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(cout,cout,3,padding=dilation,dilation=dilation,bias=False),
            nn.BatchNorm1d(cout))
        self.shortcut = nn.Identity() if cin == cout else nn.Conv1d(cin,cout,1)
    def forward(self,x): return F.gelu(self.conv(x)+self.shortcut(x))

class TinyTCN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.blocks = nn.Sequential(TemporalBlock(84,48,1),
                                    TemporalBlock(48,64,2),
                                    TemporalBlock(64,96,4))
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1),nn.Flatten(),
                                  nn.Dropout(0.3),nn.Linear(96,n_classes))
    def forward(self,x):
        b,c,t,j,d = x.shape
        return self.head(self.blocks(x.permute(0,2,1,3,4).reshape(b,t,c*j*d).transpose(1,2)))

class Tiny2p1Block(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv3d(cin,cout,(1,3,3),padding=(0,1,1),bias=False),
            nn.BatchNorm3d(cout),nn.GELU(),
            nn.Conv3d(cout,cout,(3,1,1),stride=(stride,1,1),padding=(1,0,0),bias=False),
            nn.BatchNorm3d(cout))
        self.skip = (nn.Identity() if cin==cout and stride==1 else
                     nn.Sequential(nn.Conv3d(cin,cout,1,stride=(stride,1,1),bias=False),
                                   nn.BatchNorm3d(cout)))
    def forward(self,x): return F.gelu(self.f(x)+self.skip(x))

class TinyCNN(nn.Module):
    def __init__(self,n_classes=N_CLASSES):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv3d(7,16,3,padding=1,bias=False),nn.BatchNorm3d(16),nn.GELU())
        self.net = nn.Sequential(Tiny2p1Block(16,24),Tiny2p1Block(24,48,2),
                                 Tiny2p1Block(48,72,2))
        self.head = nn.Sequential(nn.AdaptiveAvgPool3d(1),nn.Flatten(),nn.Dropout(0.3),
                                   nn.Linear(72,n_classes))
    def forward(self,x):return self.head(self.net(self.stem(x)))

# Graph node order: L elbow, R elbow, L wrist, R wrist
_G = torch.eye(4,dtype=torch.float32)
for a,b in [(0,2),(1,3),(0,1),(2,3)]:
    _G[a,b]=_G[b,a]=1.
_degree = _G.sum(dim=1)
_G = (_degree.rsqrt()[:,None] * _G * _degree.rsqrt()[None,:])

class STGCNBlock(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.spatial = nn.Linear(cin,cout,bias=False)
        self.temporal = nn.Sequential(nn.Conv2d(cout,cout,(5,1),stride=(stride,1),padding=(2,0),bias=False),
                                      nn.BatchNorm2d(cout))
        self.shortcut = (nn.Identity() if cin==cout and stride==1 else
                          nn.Sequential(nn.Conv2d(cin,cout,1,stride=(stride,1),bias=False),
                                        nn.BatchNorm2d(cout)))
    def forward(self,x,adj):  # x (B,C,T,J)
        g = torch.einsum('ij,bctj->bcti',adj,x)
        z = self.spatial(g.permute(0,2,3,1)).permute(0,3,1,2)
        return F.gelu(self.temporal(z) + self.shortcut(x))

class TinySTGCN(nn.Module):
    def __init__(self,n_classes=N_CLASSES):
        super().__init__()
        self.register_buffer('adj',_G.clone())
        self.first = nn.Conv2d(21,32,1)
        self.blocks = nn.ModuleList([STGCNBlock(32,48),STGCNBlock(48,64,2),
                                     STGCNBlock(64,80,2)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),
                                   nn.Dropout(0.3),nn.Linear(80,n_classes))
    def forward(self,x):
        b,c,t,j,d=x.shape
        z=x.permute(0,1,4,2,3).reshape(b,c*d,t,j)
        z=self.first(z)
        for block in self.blocks: z=block(z,self.adj)
        return self.head(z)



class ResidualSTGCNBlock(nn.Module):
    """Two residual paths with separate graph/temporal BN, small-data dropout.

    Input and output: (B,C,T,4). Fixed adjacency is normalized and symmetric.
    Uses MatMul/Conv/BatchNorm/Add/GELU: simple operators for ONNX Runtime Web.
    """
    def __init__(self, cin, cout, stride=1, dilation=1, dropout=0.10):
        super().__init__()
        assert stride in (1, 2) and dilation >= 1
        self.graph = nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=1, bias=False),
            nn.BatchNorm2d(cout),
        )
        self.graph_skip = (nn.Identity() if cin == cout else nn.Sequential(
            nn.Conv2d(cin, cout, kernel_size=1, bias=False),
            nn.BatchNorm2d(cout),
        ))
        self.temporal = nn.Sequential(
            nn.Conv2d(cout, cout, kernel_size=(5, 1),
                      stride=(stride, 1), padding=(2*dilation, 0),
                      dilation=(dilation, 1), bias=False),
            nn.BatchNorm2d(cout),
            nn.Dropout(p=dropout),
        )
        self.temporal_skip = (nn.Identity() if stride == 1 else nn.Sequential(
            nn.Conv2d(cout, cout, kernel_size=1, stride=(stride, 1), bias=False),
            nn.BatchNorm2d(cout),
        ))

    def forward(self, x, adj):
        # x @ adj mixes the four joints without dynamic learnable graph parameters.
        g = torch.matmul(x, adj)
        z = F.gelu(self.graph(g) + self.graph_skip(x))
        return F.gelu(self.temporal(z) + self.temporal_skip(z))


class ResidualSTGCN(nn.Module):
    """Six-block, regularized ST-GCN; same 7-channel input and 3-logit output."""
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()
        self.register_buffer('adj', _G.clone())
        self.stem = nn.Sequential(
            nn.Conv2d(21, 32, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
        )
        self.blocks = nn.ModuleList([
            ResidualSTGCNBlock(32,  48, stride=1, dilation=1, dropout=0.10),
            ResidualSTGCNBlock(48,  64, stride=1, dilation=1, dropout=0.10),
            ResidualSTGCNBlock(64,  64, stride=2, dilation=2, dropout=0.10),
            ResidualSTGCNBlock(64,  96, stride=1, dilation=1, dropout=0.12),
            ResidualSTGCNBlock(96, 128, stride=2, dilation=2, dropout=0.15),
            ResidualSTGCNBlock(128,128, stride=1, dilation=1, dropout=0.15),
        ])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(128, 64), nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        b, c, t, j, d = x.shape
        if c != 7 or j != 4 or d != 3:
            raise ValueError(f'Expected (B,7,T,4,3); got {tuple(x.shape)}')
        z = x.permute(0, 1, 4, 2, 3).reshape(b, c*d, t, j)
        z = self.stem(z)
        for block in self.blocks:
            z = block(z, self.adj)
        return self.head(z)

class TinyTransformer(nn.Module):
    def __init__(self,n_classes=N_CLASSES,d_model=64):
        super().__init__()
        self.embed = nn.Linear(84,d_model)
        self.pos = nn.Parameter(torch.randn(1,CLIP_FRAMES,d_model)*0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model,nhead=4,
                          dim_feedforward=128,dropout=0.1,batch_first=True,
                          activation='gelu',norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2,enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Dropout(0.3),nn.Linear(d_model,n_classes))
    def forward(self,x):
        b,c,t,j,d=x.shape
        z=self.embed(x.permute(0,2,1,3,4).reshape(b,t,c*j*d))
        if t > self.pos.shape[1]:
            raise ValueError('Sequence too long; increase CLIP_FRAMES and retrain.')
        z=self.encoder(z + self.pos[:,:t])
        return self.head(self.norm(z).mean(dim=1))

class TemporalDilatedNeighborhoodAttention(nn.Module):
    """Temporal local self-attention at offsets [-2,-1,0,1,2] x dilation.
    Attention is independent for each of the 4 joint tokens (joint mixing is separate).
    """
    def __init__(self,dim=48,n_heads=4,kernel=5,dilation=1):
        super().__init__()
        assert dim%n_heads==0 and kernel%2==1
        self.dim,self.heads,self.kernel,self.dilation=dim,n_heads,kernel,dilation
        self.qkv=nn.Linear(dim,3*dim)
        self.proj=nn.Linear(dim,dim)
    def forward(self,x):  # (B,T,J,D)
        b,t,j,d=x.shape
        z=x.permute(0,2,1,3).reshape(b*j,t,d)
        q,k,v=self.qkv(z).chunk(3,dim=-1)
        dh=d//self.heads
        q=q.view(b*j,t,self.heads,dh)
        k=k.view(b*j,t,self.heads,dh)
        v=v.view(b*j,t,self.heads,dh)
        offsets=torch.arange(-(self.kernel//2),self.kernel//2+1,device=x.device)*self.dilation
        times=torch.arange(t,device=x.device)[:,None]+offsets[None,:]  # (T,K)
        valid=(times>=0)&(times<t)
        times=times.clamp(0,t-1)
        kn=k[:,times]        # (B*J,T,K,H,Dh)
        vn=v[:,times]
        scores=(q[:,:,None,:,:]*kn).sum(-1)/math.sqrt(dh)    # (B*J,T,K,H)
        scores=scores.masked_fill(~valid[None,:,:,None],float('-inf'))
        attn=scores.softmax(dim=2)
        out=(attn[:,:,:,:,None]*vn).sum(2).reshape(b*j,t,d)
        return self.proj(out).reshape(b,j,t,d).permute(0,2,1,3)

class DiNATStyleBlock(nn.Module):
    def __init__(self,dim=48,dilation=1):
        super().__init__()
        self.joint_norm=nn.LayerNorm(dim)
        self.joint_attn=nn.MultiheadAttention(dim,4,batch_first=True,dropout=0.1)
        self.time_norm=nn.LayerNorm(dim)
        self.time_attn=TemporalDilatedNeighborhoodAttention(dim,4,kernel=5,dilation=dilation)
        self.ffn_norm=nn.LayerNorm(dim)
        self.ffn=nn.Sequential(nn.Linear(dim,dim*2),nn.GELU(),nn.Dropout(0.1),
                                nn.Linear(dim*2,dim))
    def forward(self,x):
        b,t,j,d=x.shape
        z=self.joint_norm(x).reshape(b*t,j,d)
        z=self.joint_attn(z,z,z,need_weights=False)[0].reshape(b,t,j,d)
        x=x+z
        x=x+self.time_attn(self.time_norm(x))
        return x+self.ffn(self.ffn_norm(x))

class TinyDiNAT(nn.Module):
    def __init__(self,n_classes=N_CLASSES,dim=48):
        super().__init__()
        self.embed = nn.Linear(21,dim)
        self.joint_pos = nn.Parameter(torch.randn(1,1,4,dim)*0.02)
        self.blocks = nn.Sequential(DiNATStyleBlock(dim,1),
                                    DiNATStyleBlock(dim,2),
                                    DiNATStyleBlock(dim,4))
        self.norm=nn.LayerNorm(dim)
        self.head=nn.Sequential(nn.Dropout(0.3),nn.Linear(dim,n_classes))
    def forward(self,x):
        b,c,t,j,d=x.shape
        z=x.permute(0,2,3,1,4).reshape(b,t,j,c*d)
        z=self.blocks(self.embed(z)+self.joint_pos)
        return self.head(self.norm(z).mean(dim=(1,2)))



class ResidualDiNATBlock(nn.Module):
    """Pre-norm joint/temporal attention + FFN; learnable residual scales.

    The attention operates on a small (T=50, J=4) skeleton grid, not on images.
    Linear/LayerNorm/MatMul/Softmax/Gather operations should be checked after
    ONNX export in the target onnxruntime-web version before deployment.
    """
    def __init__(self, dim=96, n_heads=4, dilation=1, dropout=0.10):
        super().__init__()
        self.joint_norm = nn.LayerNorm(dim)
        self.joint_attn = nn.MultiheadAttention(dim, n_heads, dropout=dropout,
                                                 batch_first=True)
        self.time_norm = nn.LayerNorm(dim)
        self.time_attn = TemporalDilatedNeighborhoodAttention(
            dim=dim, n_heads=n_heads, kernel=5, dilation=dilation)
        self.ffn_norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(2 * dim, dim))
        self.resid_drop = nn.Dropout(dropout)
        # LayerScale avoids large early residual updates in a deeper network.
        self.joint_scale = nn.Parameter(torch.full((dim,), 0.1))
        self.time_scale = nn.Parameter(torch.full((dim,), 0.1))
        self.ffn_scale = nn.Parameter(torch.full((dim,), 0.1))

    def forward(self, x):  # (B,T,4,D)
        b, t, j, d = x.shape
        z = self.joint_norm(x).reshape(b * t, j, d)
        z = self.joint_attn(z, z, z, need_weights=False)[0].reshape(b, t, j, d)
        x = x + self.resid_drop(z) * self.joint_scale
        z = self.time_attn(self.time_norm(x))
        x = x + self.resid_drop(z) * self.time_scale
        z = self.ffn(self.ffn_norm(x))
        return x + self.resid_drop(z) * self.ffn_scale


class ResidualDiNAT(nn.Module):
    """Moderately deeper temporal-dilated attention for four arm joints."""
    def __init__(self, n_classes=N_CLASSES, dim=96):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(21, dim), nn.LayerNorm(dim))
        self.joint_pos = nn.Parameter(torch.randn(1, 1, 4, dim) * 0.02)
        self.blocks = nn.Sequential(*[
            ResidualDiNATBlock(dim, n_heads=4, dilation=d, dropout=0.10)
            for d in (1, 2, 4, 8)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Sequential(nn.LayerNorm(2 * dim),
                                  nn.Dropout(0.25), nn.Linear(2 * dim, dim),
                                  nn.GELU(), nn.Dropout(0.20),
                                  nn.Linear(dim, n_classes))

    def forward(self, x):
        b, c, t, j, d = x.shape
        if c != 7 or j != 4 or d != 3:
            raise ValueError(f'Expected (B,7,T,4,3), got {tuple(x.shape)}')
        z = x.permute(0, 2, 3, 1, 4).reshape(b, t, j, c * d)
        z = self.norm(self.blocks(self.embed(z) + self.joint_pos))
        pooled = torch.cat((z.mean(dim=(1,2)), z.amax(dim=(1,2))), dim=-1)
        return self.head(pooled)

MODEL_ZOO = {'tcn':TinyTCN,'tiny_cnn':TinyCNN,'stgcn':TinySTGCN,
             'stgcn_residual':ResidualSTGCN,
             'transformer':TinyTransformer,'tiny_dinat':TinyDiNAT,
              'dinat_residual':ResidualDiNAT,
             'original_v12_cnn':PerfectPunchCNN}

def create_model(name):
    if name not in MODEL_ZOO: raise ValueError(f'Unknown {name}; choices: {list(MODEL_ZOO)}')
    return MODEL_ZOO[name](num_classes=N_CLASSES) if name == 'original_v12_cnn' else MODEL_ZOO[name](n_classes=N_CLASSES)

def count_parameters(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

for name in MODEL_NAMES:
    model=create_model(name)
    print(f'{name:15}: {count_parameters(model):>9,} trainable params')

# File-size comparison (FP32 parameter weights vs serialized state_dict).
# Actual ONNX graph size is measured only after export.
def print_comparison_footprints():
    import tempfile
    print('Estimated and measured PyTorch weight sizes (untrained models):')
    for name in ('stgcn', 'stgcn_residual', 'tiny_dinat', 'dinat_residual'):
        net = create_model(name)
        n_train = count_parameters(net)
        n_total = sum(p.numel() for p in net.parameters())
        buffers = sum(t.numel() * t.element_size() for t in net.buffers())
        param_bytes = sum(p.numel() * p.element_size() for p in net.parameters())
        with tempfile.TemporaryDirectory() as tmp:
            path = os.path.join(tmp, 'weights.pt')
            torch.save(net.state_dict(), path)
            disk = os.path.getsize(path)
        print(f'{name:16} trainable={n_train:>9,}  FP32 params={param_bytes/1e6:.3f} MB  '
              f'buffers={buffers/1e3:.1f} KB  state_dict={disk/1e6:.3f} MB')
    print('Actual ONNX bytes depend on export/graph overhead; call export_onnx() after training.')

print_comparison_footprints()


tcn            :   101,171 trainable params
tiny_cnn       :    78,251 trainable params
stgcn          :    85,171 trainable params
stgcn_residual :   361,795 trainable params
transformer    :    75,907 trainable params
tiny_dinat     :    86,883 trainable params
dinat_residual :   472,131 trainable params
Estimated and measured PyTorch weight sizes (untrained models):
stgcn            trainable=   85,171  FP32 params=0.341 MB  buffers=3.2 KB  state_dict=0.358 MB
stgcn_residual   trainable=  361,795  FP32 params=1.447 MB  buffers=13.1 KB  state_dict=1.499 MB
tiny_dinat       trainable=   86,883  FP32 params=0.348 MB  buffers=0.0 KB  state_dict=0.368 MB
dinat_residual   trainable=  472,131  FP32 params=1.889 MB  buffers=0.0 KB  state_dict=1.920 MB
Actual ONNX bytes depend on export/graph overhead; call export_onnx() after training.


## 8. Experiment training / held-out predictions

Per fold and per architecture, use **only training videos** for gradient updates and **only validation videos** to select the best epoch via cross-entropy loss (macro-F1 is a tie-breaker). Then load the selected checkpoint and evaluate its **24 held-out test videos** once (for the current 120-video dataset). All five architectures use identical stratified splits. Cross-validation checkpoints are not final deployment models; use the full-data refit in Section 12 for deployment.


In [10]:
def loaders_for_fold(manifest, f):
    clean_ds=BoxingDataset(manifest,augment=False)
    train_ds=BoxingDataset(manifest,augment=AUGMENT_TRAIN)
    ids = f['train']
    labels=manifest.iloc[ids].label.to_numpy()
    if SAMPLER_MODE=='balanced':
        counts=np.bincount(labels,minlength=N_CLASSES)
        weights=1.0 / np.maximum(counts[labels],1)
    elif SAMPLER_MODE=='hook_x3':
        weights=np.where(labels==CLASS_NAMES.index('lead-hook'),3.,1.)
    elif SAMPLER_MODE=='shuffle':
        weights=None
    else:
        raise ValueError(SAMPLER_MODE)
    sampler=(WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
                num_samples=len(ids),replacement=True) if weights is not None else None)
    train=DataLoader(Subset(train_ds,ids.tolist()),batch_size=BATCH_SIZE,
                     shuffle=sampler is None,sampler=sampler,num_workers=NUM_WORKERS,
                     pin_memory=torch.cuda.is_available())
    def val_loader(indices):
        return DataLoader(Subset(clean_ds,indices.tolist()),batch_size=BATCH_SIZE,
                          shuffle=False,num_workers=NUM_WORKERS,pin_memory=torch.cuda.is_available())
    return train,val_loader(f['val']),val_loader(f['test'])

def run_one_epoch(model,loader,device,optimizer=None):
    training=optimizer is not None
    model.train(training)
    criterion=nn.CrossEntropyLoss(label_smoothing=0.05 if training else 0.0)
    total_loss,n=0.,0
    truth,preds=[],[]
    with torch.set_grad_enabled(training):
        for X,y in loader:
            X=X.to(device,non_blocking=True); y=y.to(device,non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            logits=model(X)
            loss=criterion(logits,y)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.)
                optimizer.step()
            total_loss+=loss.item()*len(y);n+=len(y)
            truth.extend(y.detach().cpu().tolist())
            preds.extend(logits.argmax(1).detach().cpu().tolist())
    if not n: raise ValueError('Empty DataLoader in epoch')
    return {'loss':total_loss/n,'acc':accuracy_score(truth,preds),
            'macro_f1':f1_score(truth,preds,labels=list(range(N_CLASSES)),
                               average='macro',zero_division=0)}

@torch.no_grad()
def collect_predictions(model,loader,device):
    model.eval()
    labels,preds,probs=[],[],[]
    for X,y in loader:
        out=model(X.to(device)).softmax(1).cpu().numpy()
        labels.extend(y.numpy().tolist())
        preds.extend(out.argmax(1).tolist())
        probs.extend(out.tolist())
    return np.asarray(labels),np.asarray(preds),np.asarray(probs)



def train_one_fold(manifest, f, model_name, device):
    seed_everything(SEED + f['fold'])
    train_loader, val_loader, test_loader = loaders_for_fold(manifest, f)
    model = create_model(model_name).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_loss=float('inf'); best_f1=-1.; wait=0; best_epoch=None
    model_dir = os.path.join(RUN_DIR, model_name)
    os.makedirs(model_dir, exist_ok=True)
    ckpt = os.path.join(model_dir, f"fold_{f['fold']:03d}_stratified5.pt")
    for epoch in range(1, EPOCHS+1):
        tr = run_one_epoch(model, train_loader, device, optimizer)
        val = run_one_epoch(model, val_loader, device)
        scheduler.step()
        better = (val['loss'] < best_loss-1e-5 or
                  (abs(val['loss']-best_loss)<=1e-5 and val['macro_f1']>best_f1))
        if better:
            best_loss, best_f1 = val['loss'], val['macro_f1']
            best_epoch=epoch; wait=0
            torch.save({'state_dict': model.state_dict(), 'model_name': model_name,
                        'classes': CLASS_NAMES, 'clip_frames': CLIP_FRAMES, 'fps':TARGET_FPS, 'n_channels':N_CHANNELS, 'augmentation_mode':AUGMENTATION_MODE, 'pose_variant': POSE_VARIANT,
                        'fold': f['fold'], 'test_video_ids': f['test_video_ids'],
                        'validation_video_ids': manifest.iloc[f['val']].video_id.tolist(),
                        'split_strategy': 'stratified_5_fold_video_level_60fps_peak_v25',
                        'preprocessing_contract': PREPROCESS_CONTRACT,
                        'webcam_fps_aug_prob': WEBCAM_FPS_AUG_PROB,
                        'webcam_capture_fps': WEBCAM_CAPTURE_FPS,
                        'epoch': epoch}, ckpt)
        else:
            wait += 1
        if epoch==1 or epoch%10==0 or better:
            print(f"{model_name} fold={f['fold']} ep={epoch:03d} "
                  f"train_acc={tr['acc']:.3f} val_f1={val['macro_f1']:.3f} "
                  f"val_loss={val['loss']:.3f}{' *' if better else ''}")
        if wait >= PATIENCE:
            break
    saved = torch.load(ckpt, map_location=device, weights_only=True)
    model.load_state_dict(saved['state_dict'])
    actual, pred, prob = collect_predictions(model, test_loader, device)
    # A SECOND held-out report with a deterministic sparse-30-FPS proxy.
    # This is diagnostic only: no real webcam or end-to-end JS tracking here.
    webcam_ds = BoxingDataset(manifest, augment=False, webcam_simulated=True)
    webcam_loader = DataLoader(Subset(webcam_ds, f['test'].tolist()),
                               batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    actual_web, pred_web, prob_web = collect_predictions(model, webcam_loader, device)
    assert np.array_equal(actual, actual_web)
    records=[]
    webcam_records=[]
    for i, index in enumerate(f['test']):
        row = manifest.iloc[int(index)]
        rec={'model':model_name, 'fold': f['fold'], 'video_id':row.video_id,
             'relative_path':row.relative_path, 'path':row.path,
             'class_name':row.class_name,
             'true':int(actual[i]), 'pred':int(pred[i]),
             'confidence':float(prob[i].max()),'best_epoch':best_epoch}
        for k,cls in enumerate(CLASS_NAMES):
            rec[f'p_{cls}'] = float(prob[i,k])
        records.append(rec)
        wc = dict(rec)
        wc['pred'] = int(pred_web[i])
        wc['confidence'] = float(prob_web[i].max())
        for k, cls in enumerate(CLASS_NAMES):
            wc[f'p_{cls}'] = float(prob_web[i, k])
        webcam_records.append(wc)
    return records, webcam_records, ckpt


def evaluate_stratified5(manifest, model_names=MODELS_TO_RUN, max_folds=MAX_FOLDS):
    """Evaluate fixed stratified 5-fold test sets; never use test data for early stopping."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    folds = list(iter_stratified5_folds(manifest, max_folds=max_folds))
    print(f'Running {len(folds)} folds x {len(model_names)} model(s); '
          f'{sum(len(f["test"]) for f in folds)}/{len(manifest)} test videos.')
    all_records=[]; webcam_records=[]; checkpoints=[]
    for model_name in model_names:
        print('\n'+'='*75+'\nMODEL:',model_name)
        for f in folds:
            records, simulated_records, ckpt = train_one_fold(manifest, f, model_name, device)
            all_records.extend(records)
            webcam_records.extend(simulated_records)
            checkpoints.append({'model':model_name, 'fold':f['fold'],
                                'num_test_videos':len(f['test']), 'checkpoint':ckpt})
            pd.DataFrame(all_records).to_csv(
                os.path.join(RUN_DIR,'stratified5_predictions_partial.csv'),index=False)
    predictions=pd.DataFrame(all_records)
    predictions.to_csv(os.path.join(RUN_DIR, 'stratified5_predictions.csv'),index=False)
    pd.DataFrame(webcam_records).to_csv(
        os.path.join(RUN_DIR, 'stratified5_webcam_proxy_predictions.csv'), index=False)
    pd.DataFrame(checkpoints).to_csv(os.path.join(RUN_DIR, 'stratified5_checkpoints.csv'),index=False)
    print('Saved predictions/checkpoints to', RUN_DIR)
    return predictions

# manifest = build_manifest()
# check_folds(manifest)
# predictions = evaluate_stratified5(manifest, MODELS_TO_RUN)  # four default candidates

## 9. Results, confusion matrix, per-video mistakes

In [11]:
def summarize_stratified5(predictions, show_plots=True, filename='stratified5_summary.csv'):
    if predictions.empty:
        raise ValueError('No predictions yet. Run evaluate_stratified5().')
    summary=[]
    for name, frame in predictions.groupby('model', sort=False):
        yt=frame.true.to_numpy(dtype=int); yp=frame.pred.to_numpy(dtype=int)
        per_fold = frame.groupby('fold', sort=True).apply(
            lambda g: pd.Series({'accuracy': accuracy_score(g.true,g.pred),
              'macro_f1': f1_score(g.true,g.pred,labels=list(range(N_CLASSES)),
                                   average='macro',zero_division=0),
              'test_videos':len(g)}), include_groups=False)
        row={'model':name, 'tested_videos':len(frame), 'folds':len(per_fold),
             'pooled_accuracy':accuracy_score(yt,yp),
             'pooled_macro_f1':f1_score(yt,yp,labels=list(range(N_CLASSES)),
                                      average='macro',zero_division=0),
             'mean_fold_accuracy':per_fold.accuracy.mean(),
             'mean_fold_macro_f1':per_fold.macro_f1.mean()}
        summary.append(row)
        print('\n',name,'\n',pd.DataFrame([row]).to_string(index=False))
        print('\nPer-fold:\n',per_fold.to_string())
        print('\nClass breakdown:\n',classification_report(
            yt,yp,labels=list(range(N_CLASSES)),target_names=CLASS_NAMES,
            digits=3,zero_division=0))
        if show_plots:
            disp=ConfusionMatrixDisplay(confusion_matrix(
                yt,yp,labels=list(range(N_CLASSES))),display_labels=CLASS_NAMES)
            disp.plot(cmap='Blues',values_format='d',xticks_rotation=20)
            plt.title(f'{name}: video-held-out pooled predictions')
            plt.tight_layout();plt.show()
    table=pd.DataFrame(summary)
    table.to_csv(os.path.join(RUN_DIR,filename),index=False)
    return table

# summary = summarize_stratified5(predictions)
# errors = predictions[predictions.true != predictions.pred]

## 10. Sanity checks: feature contract, model gradients, and stratified fold coverage

Runs synthetic tests **without Google Drive videos**. The split test checks five disjoint, stratified folds with each video held out exactly once; gradient checks verify the five lightweight architectures. Real-video extraction and ONNX should still be tested in your own Colab environment.


In [12]:
def smoke_tests():
    seed_everything(7)
    raw = np.zeros((CLIP_FRAMES,33,4),dtype=np.float32)
    raw[:,:,:3] = np.random.normal(0,0.3,(CLIP_FRAMES,33,3))
    x = torch.from_numpy(build_tensor(raw)).unsqueeze(0)
    assert x.shape == (1,7,CLIP_FRAMES,4,3) and torch.isfinite(x).all()
    for name in MODEL_NAMES:
        net=create_model(name)
        out=net(x)
        assert out.shape==(1,N_CLASSES) and torch.isfinite(out).all(),name
        out.square().mean().backward()
        assert any(p.grad is not None for p in net.parameters()),name
        print(name, 'OK | params=',count_parameters(net))
    # Synthetic dataset that mirrors your real 40 + 40 + 40 videos.
    synthetic=pd.DataFrame([{'video_id':f'{CLASS_NAMES[c]}/clip_{i:03d}',
                             'class_name':CLASS_NAMES[c], 'label':c}
                             for c in range(N_CLASSES) for i in range(40)])
    tests=[]
    folds=list(iter_stratified5_folds(synthetic, seed=13, max_folds=None))
    assert len(folds)==5
    for f in folds:
        train,val,test=set(f['train']),set(f['val']),set(f['test'])
        assert len(train)==81 and len(val)==15 and len(test)==24
        assert train.isdisjoint(val) and train.isdisjoint(test) and val.isdisjoint(test)
        assert set(synthetic.iloc[f['train']].label)==set(range(N_CLASSES))
        assert np.bincount(synthetic.iloc[f['test']].label.to_numpy(), minlength=N_CLASSES).tolist()==[8,8,8]
        assert np.bincount(synthetic.iloc[f['val']].label.to_numpy(), minlength=N_CLASSES).tolist()==[5,5,5]
        assert len(f['test_video_ids'])==len(f['test'])
        tests.extend(f['test'])
    assert sorted(tests)==list(range(len(synthetic)))
    # Segment-contract smoke test. Earliest qualifying 4-joint speed peak wins.
    fake = np.zeros((85, 33, 4), dtype=np.float32)
    # Dramatic jump at frame 60 (arms only), stable elsewhere.
    fake[60:, VELOCITY_LM, 0] = 3.0
    peak, _ = find_peak_velocity_frame(fake)
    assert abs(peak - 60) <= 2, peak  # moving-average plateau can shift ±2
    proxy = simulate_webcam_30_to_60(fake[:CLIP_FRAMES])
    assert proxy.shape == (CLIP_FRAMES,33,4) and np.isfinite(proxy).all()
    print('All smoke tests passed: peak selection, webcam proxy, 5 stratified folds, 120/120 test coverage, no overlap.')

# smoke_tests()

def audit_v25_data_files(manifest, n_per_class=2):
    print('v25 peak-window data compatibility audit (sampled files):')
    for label in CLASS_NAMES:
        sub=manifest.loc[manifest.class_name.eq(label)].head(n_per_class)
        if sub.empty:
            raise ValueError(f'Missing class directory/data: {label}')
        for row in sub.itertuples():
            raw=np.load(row.path)
            a=prepare_raw_clip(raw)
            f=build_tensor(a)
            assert a.shape==(CLIP_FRAMES,33,4), (row.path,a.shape)
            assert f.shape==(7,CLIP_FRAMES,4,3), (row.path,f.shape)
            assert np.isfinite(f).all(), f'Non-finite feature values: {row.path}'
            print(f'  {row.relative_path}: raw={tuple(raw.shape)} -> tensor={tuple(f.shape)}')
    print('All sampled clips follow the v25 data-shape contract.')

In [13]:
# No Drive videos required for this check; run after model/data setup.
smoke_tests()
# When segmentation has completed:
# manifest = build_manifest()
# check_folds(manifest)
# audit_v25_data_files(manifest)

tcn OK | params= 101171
tiny_cnn OK | params= 78251
stgcn OK | params= 85171
stgcn_residual OK | params= 361795
transformer OK | params= 75907
tiny_dinat OK | params= 86883
dinat_residual OK | params= 472131
All smoke tests passed: peak selection, webcam proxy, 5 stratified folds, 120/120 test coverage, no overlap.


## 11. Run **v25 Lite peak-based stratified five-fold CV**

`RUN_EXPERIMENTS=True`, `N_SPLITS=5`, `MAX_FOLDS=None`, and the four default models produce **20 fold-training jobs**. Each architecture is trained from scratch on exactly the same split indices, using the **new v25 peak-based clips** (not cached reversal-based v23 clips). Training augmentation includes optional 30→60 FPS webcam simulation; held-out clean metrics and separate webcam-proxy held-out metrics are reported. Saving both metrics does not create an independent new sample set; they share the same held-out videos. The split is by video, not person.


In [14]:
if RUN_EXPERIMENTS:
    manifest = build_manifest()
    check_folds(manifest)
    audit_v25_data_files(manifest)
    predictions = evaluate_stratified5(manifest, MODELS_TO_RUN)
    summary = summarize_stratified5(predictions)
    proxy_predictions = pd.read_csv(os.path.join(RUN_DIR, 'stratified5_webcam_proxy_predictions.csv'))
    proxy_summary = summarize_stratified5(proxy_predictions, show_plots=False,
                 filename='stratified5_webcam_proxy_summary.csv')
    print('Standard held-out vs synthetic 30-FPS proxy:')
    print(summary[['model','pooled_macro_f1']].merge(
        proxy_summary[['model','pooled_macro_f1']], on='model',
        suffixes=('_clean','_webcam_proxy')).to_string(index=False))
    print('Results:', RUN_DIR)
else:
    print('Training disabled. Set RUN_EXPERIMENTS=True in config and rerun this cell to start.')

Video counts: {'jab': 40, 'lead-hook': 40, 'rear-uppercut': 40} | Total: 120
Fold 000 | train= 81 val= 15 test= 24 | test_classes={'jab': 8, 'lead-hook': 8, 'rear-uppercut': 8}
Fold 001 | train= 81 val= 15 test= 24 | test_classes={'jab': 8, 'lead-hook': 8, 'rear-uppercut': 8}
Fold 002 | train= 81 val= 15 test= 24 | test_classes={'jab': 8, 'lead-hook': 8, 'rear-uppercut': 8}
Fold 003 | train= 81 val= 15 test= 24 | test_classes={'jab': 8, 'lead-hook': 8, 'rear-uppercut': 8}
Fold 004 | train= 81 val= 15 test= 24 | test_classes={'jab': 8, 'lead-hook': 8, 'rear-uppercut': 8}
Test coverage: 120/120 unique videos across 5 folds.
v25 peak-window data compatibility audit (sampled files):
  jab/jab_0001.npy: raw=(50, 33, 4) -> tensor=(7, 50, 4, 3)
  jab/jab_0002.npy: raw=(50, 33, 4) -> tensor=(7, 50, 4, 3)
  lead-hook/hook_0001.npy: raw=(50, 33, 4) -> tensor=(7, 50, 4, 3)
  lead-hook/hook_0002.npy: raw=(50, 33, 4) -> tensor=(7, 50, 4, 3)
  rear-uppercut/uppercut_0001.npy: raw=(50, 33, 4) -> tens

KeyboardInterrupt: 

## 12. Final deployment model training (only after CV model selection)

Full five-fold evaluation comes first. Review clean AND synthetic webcam-proxy macro-F1 plus per-class recall and confusion matrices before choosing `FINAL_MODEL_NAME`. Set `RUN_FINAL_REFIT=True` only after selecting your architecture. The refit uses all 120 labeled examples and the median best validation epoch from the 5 folds; its in-sample accuracy is *not* independent held-out accuracy. The JavaScript model needs a new ONNX export with matching input/output tensor names. For actual browser accuracy, run a separate webcam capture test.


In [15]:
def fit_final_model(manifest, model_name, epochs, filename=None):
    """Final refit on all available labeled videos after CV model selection."""
    if model_name not in MODEL_ZOO: raise ValueError(model_name)
    if int(epochs) < 1: raise ValueError('epochs must be positive')
    seed_everything(SEED + 50000)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = create_model(model_name).to(device)
    full_ds = BoxingDataset(manifest, augment=AUGMENT_TRAIN)
    labels = manifest.label.to_numpy(dtype=int)
    if SAMPLER_MODE == 'hook_x3':
        weights = np.where(labels==CLASS_NAMES.index('lead-hook'),3.,1.)
    elif SAMPLER_MODE == 'balanced':
        counts = np.bincount(labels,minlength=N_CLASSES)
        weights = 1. / np.maximum(counts[labels],1)
    elif SAMPLER_MODE == 'shuffle':
        weights = None
    else: raise ValueError(SAMPLER_MODE)
    sampler = WeightedRandomSampler(torch.as_tensor(weights,dtype=torch.double),
                                    num_samples=len(labels),replacement=True) if weights is not None else None
    loader = DataLoader(full_ds,batch_size=BATCH_SIZE,shuffle=sampler is None,
                        sampler=sampler,num_workers=NUM_WORKERS)
    optim = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)
    for epoch in range(1,int(epochs)+1):
        metrics = run_one_epoch(model,loader,device,optim)
        scheduler.step()
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f'FINAL {model_name} ep={epoch}/{epochs}: acc={metrics["acc"]:.3f} loss={metrics["loss"]:.3f}')
    out = filename or os.path.join(RUN_DIR,f'FINAL_{model_name}.pt')
    torch.save({'state_dict':model.cpu().state_dict(),'model_name':model_name,
                'classes':CLASS_NAMES,'clip_frames':CLIP_FRAMES,'fps':TARGET_FPS,
                'n_channels':N_CHANNELS,'training_videos':len(manifest),'epochs':int(epochs),
                'augmentation_mode':AUGMENTATION_MODE, 'pose_variant': POSE_VARIANT,
                'preprocessing_contract':PREPROCESS_CONTRACT,
                'webcam_fps_aug_prob':WEBCAM_FPS_AUG_PROB, 'webcam_capture_fps':WEBCAM_CAPTURE_FPS,
                'role':'final_full_data_refit'},out)
    print('Final model saved to:',out)
    return out

# Example after FULL cross-validation:
# best_name = 'stgcn'   # use actual results
# cv = pd.read_csv(os.path.join(RUN_DIR,'stratified5_predictions.csv'))
# best_epochs = cv.loc[cv.model.eq(best_name),'best_epoch'].to_numpy()
# final_epochs = int(np.median(best_epochs))
# final_ckpt = fit_final_model(build_manifest(),best_name,final_epochs)

## 13. Offline real-video prediction and ONNX export

`predict_video()` uses the same MediaPipe Lite world-landmark normalization, 60-FPS resampling, **four-joint / five-frame smoothed peak**, 50-frame window, and seven-channel features as training (no augmentation at inference). `export_onnx()` exports input `skeleton_volume` `(B,7,50,4,3)` and output `class_logits` `(B,3)`. The team's current JS runtime instead expects `input`/`logits`; change JS inference keys or export names before integration. The ONNX architecture must be supported by the browser's selected ONNX Runtime Web provider, particularly for attention-based variants.


In [16]:
def load_checkpoint(checkpoint, device=None):
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    meta = torch.load(checkpoint, map_location='cpu', weights_only=True)
    assert meta['clip_frames'] == CLIP_FRAMES and meta['classes'] == CLASS_NAMES
    assert meta.get('fps',TARGET_FPS) == TARGET_FPS and meta.get('n_channels',7) == N_CHANNELS
    if meta.get('preprocessing_contract') != PREPROCESS_CONTRACT:
        raise ValueError(f'Wrong checkpoint preprocessing: expected {PREPROCESS_CONTRACT}, got {meta.get("preprocessing_contract")}')
    if meta.get('pose_variant') is not None and meta['pose_variant'] != POSE_VARIANT:
        raise ValueError(f'Checkpoint trained using {meta["pose_variant"]} pose; active preprocessing is {POSE_VARIANT}')
    net = create_model(meta['model_name']).to(device)
    net.load_state_dict(meta['state_dict'])
    return net.eval(), meta, device

@torch.no_grad()
def predict_video(video_path, checkpoint):
    net, meta, device = load_checkpoint(checkpoint)
    clip = extract_best_clip(video_path)
    if clip is None: return None
    x = torch.from_numpy(build_tensor(prepare_raw_clip(clip))).unsqueeze(0).to(device)
    logits = net(x)
    probs = logits.softmax(1)[0].cpu().numpy()
    return {'predicted_label':CLASS_NAMES[int(probs.argmax())],
            'confidence':float(probs.max()),
            'probabilities':{name:float(prob) for name,prob in zip(CLASS_NAMES,probs)},
            'model':meta['model_name']}

def export_onnx(checkpoint, filename=None, opset=17):
    net, meta, _ = load_checkpoint(checkpoint,torch.device('cpu'))
    out = filename or os.path.join(RUN_DIR,f"{meta['model_name']}_final.onnx")
    dummy = torch.randn(1,N_CHANNELS,CLIP_FRAMES,N_ARM_JOINTS,3)
    torch.onnx.export(net,dummy,out,opset_version=opset,do_constant_folding=True,
                      input_names=['skeleton_volume'],output_names=['class_logits'],
                      dynamic_axes={'skeleton_volume':{0:'batch'},'class_logits':{0:'batch'}})
    print(f'ONNX saved: {out} ({os.path.getsize(out)/1e6:.3f} MB / {os.path.getsize(out)/1048576:.3f} MiB)')
    return out

# predict_video('/content/drive/MyDrive/.../jab/jab_0001.mp4', final_ckpt)
# export_onnx(final_ckpt)

## 14. Important evaluation and deployment notes

- **Never reuse old processed `.npy` clips** from v21/v22/v23/v24. v25 has a new extraction folder and JSON preprocessing contract. Old reversal-based checkpoints are incompatible with the new peak-based distribution.
- Data augmentation (including the default random 30→60 FPS proxy) is **training-only**. Clean validation/test splits are untouched. The additional deterministic webcam-proxy test is held out by video but is still **synthetic**, not actual website accuracy.
- Whole-video extraction searches for one global speed peak after the 50-frame minimum. The website's `segmentClip(startT,endT)` first narrows the search window using a punch detector, and its `snapshotClip()` fallback may select different samples. Matching the core peak-velocity algorithm **does not guarantee identical boundaries**.
- `web/pipeline.js` needs the agreed **4-joint + 5-frame** settings; v25 feature channel 5 already uses the full 3D speed denominator. JavaScript currently uses `sqrt(vx²+vz²)` and must change to `sqrt(vx²+vy²+vz²)`. Verify all seven channels numerically using a saved skeleton clip.
- Keep 60 FPS as the resampling target, not a claim that the browser or webcam processes 60 real frames/second. The live JS code estimates average processing FPS, which can drift and obscure dropped/irregular frames.
- Your browser currently requires >=50 resampled frames and returns null otherwise; this offline notebook front-pads short videos. Avoid short clips in deployment or explicitly align the two policies.
- Stratified CV is **video-level, not boxer-disjoint**. It is not an estimate of performance on unseen people. Use a fresh person-disjoint webcam set to select the final model for real use.
- ONNX export default names are `skeleton_volume`/`class_logits`, whereas current website expects `input`/`logits`. Ensure names and provider support match the actual ONNX file loaded; do not just replace the 21 MB CNN weights with a different architecture.


## 15. Optional final full-data refit (no Gradio)

After comparing the v25 five-fold clean and webcam-proxy results, select `FINAL_MODEL_NAME` (`stgcn`, `stgcn_residual`, `tiny_dinat`, or `dinat_residual`) and set `RUN_FINAL_REFIT=True`. Re-run this section, then export the final checkpoint to ONNX using `export_onnx(final_ckpt)` in Section 13. No Gradio or legacy webcam demo cells are included; the project's deployment target is JavaScript MediaPipe + ONNX Runtime Web.


In [17]:
if RUN_FINAL_REFIT:
    # SECTION 15 — opt-in deployment refit AFTER checking complete v25 CV results
    import os
    import numpy as np
    import pandas as pd
    from torch.utils.data import DataLoader

    manifest = build_manifest()
    assert len(manifest) == 120, f"Expected the same 120-video dataset; got {len(manifest)}"
    cv_path = os.path.join(RUN_DIR, 'stratified5_predictions.csv')
    if os.path.isfile(cv_path):
        cv = pd.read_csv(cv_path)
        selected_folds = (cv.loc[cv['model'] == FINAL_MODEL_NAME, ['fold', 'best_epoch']]
                         .drop_duplicates(subset=['fold']).sort_values('fold'))
        assert selected_folds['fold'].nunique() == N_SPLITS, (
            f"Expected {N_SPLITS} completed v25 {FINAL_MODEL_NAME} folds, got {len(selected_folds)}. "
            f"Run all CV folds for this candidate, or remove stale {cv_path}."
        )
        FINAL_EPOCHS = max(1, int(round(selected_folds['best_epoch'].median())))
        print(f'Lite {FINAL_MODEL_NAME} CV best epochs:', selected_folds['best_epoch'].tolist())
        print(f'Lite {FINAL_MODEL_NAME} CV pooled held-out accuracy:',
              f"{(cv.loc[cv.model == FINAL_MODEL_NAME, 'true'] == cv.loc[cv.model == FINAL_MODEL_NAME, 'pred']).mean():.2%}")
    else:
        raise FileNotFoundError(f'No v25 CV predictions at {cv_path}; run all 5 folds first.')
    print('Selected final architecture:', FINAL_MODEL_NAME)
    print('Full-data training epochs:', FINAL_EPOCHS)
    print('Training config:', 'augmentation=', AUGMENTATION_MODE, 'sampler=', SAMPLER_MODE)

    final_ckpt = fit_final_model(
        manifest,
        model_name=FINAL_MODEL_NAME,
        epochs=FINAL_EPOCHS,
        filename=os.path.join(RUN_DIR, f'FINAL_{FINAL_MODEL_NAME}_full120.pt'),
    )

    # Only an IN-SAMPLE sanity check; this is NOT held-out generalization accuracy.
    net, meta, device = load_checkpoint(final_ckpt)
    training_loader = DataLoader(
        BoxingDataset(manifest, augment=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    training_metrics = run_one_epoch(net, training_loader, device)
    print(f"IN-SAMPLE final training accuracy: {training_metrics['acc']:.2%}")
    print(f"IN-SAMPLE final training macro-F1: {training_metrics['macro_f1']:.4f}")
    print('NOTE: In-sample final training scores are NOT generalization accuracy.')
    print('Final checkpoint:', final_ckpt)
else:
    print('CV-only run: inspect the v25 clean + webcam-proxy summaries; choose FINAL_MODEL_NAME, then enable RUN_FINAL_REFIT.')


Video counts: {'jab': 40, 'lead-hook': 40, 'rear-uppercut': 40} | Total: 120
Lite dinat_residual CV best epochs: [116, 13, 86, 83, 62]
Lite dinat_residual CV pooled held-out accuracy: 84.17%
Selected final architecture: dinat_residual
Full-data training epochs: 83
Training config: augmentation= v12 sampler= hook_x3
FINAL dinat_residual ep=1/83: acc=0.542 loss=1.017
FINAL dinat_residual ep=10/83: acc=0.775 loss=0.660
FINAL dinat_residual ep=20/83: acc=0.825 loss=0.606
FINAL dinat_residual ep=30/83: acc=0.875 loss=0.432
FINAL dinat_residual ep=40/83: acc=0.983 loss=0.267
FINAL dinat_residual ep=50/83: acc=0.925 loss=0.301
FINAL dinat_residual ep=60/83: acc=0.942 loss=0.309
FINAL dinat_residual ep=70/83: acc=0.975 loss=0.263
FINAL dinat_residual ep=80/83: acc=0.950 loss=0.299
FINAL dinat_residual ep=83/83: acc=0.933 loss=0.334
Final model saved to: /content/drive/MyDrive/newPerfectPunchData/stratified5_lite_peak_v25/FINAL_dinat_residual_full120.pt
IN-SAMPLE final training accuracy: 95.83%

## 16. JavaScript handoff checklist (website code **not modified**)

To align `perfect-punch-test-main/web/pipeline.js` with **this** notebook, the teammate should update:

```javascript
export const CLIP_FRAMES = 50;
export const TARGET_FPS = 60;
export const VEL_WINDOW = 5;                 // formerly 7
const VELOCITY_LM = [13, 14, 15, 16];       // formerly included shoulders
// ch5: vx / (Math.hypot(vx, vy, vz) + 1e-8)
```

Keep `resampleFrames()` and the 50-frame window *ending at and including* the smoothed peak. Verify the website's punch detector window selection does not truncate the true peak or omit the required lead-in. JS currently uses `segmentClip()` and a different fallback; do not assume it matches an offline whole-video peak search.

To integrate the exported ONNX model, update `game.js`'s ONNX input/output names to match `skeleton_volume` / `class_logits`, or explicitly export with `input` / `logits`. Preserve class ordering. Test the output on exactly the same saved 50-frame skeleton tensor in Python and JS before evaluating webcam performance. The JS website ZIP is **not edited** by this notebook.
